# EDA — Fraud_Data.csv
Exploratory analysis covering data cleaning, univariate/bivariate distributions, class imbalance, and geolocation integration.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

fraud = pd.read_csv('../data/raw/Fraud_Data.csv')
ip_country = pd.read_csv('../data/raw/IpAddress_to_Country.csv')

print(fraud.shape)
fraud.head()

## 1. Data Cleaning

In [ ]:
# Missing values
print('Missing values:')
print(fraud.isnull().sum())

# Duplicates
print(f'\nDuplicates: {fraud.duplicated().sum()}')
fraud.drop_duplicates(inplace=True)

# Data types
fraud['signup_time'] = pd.to_datetime(fraud['signup_time'])
fraud['purchase_time'] = pd.to_datetime(fraud['purchase_time'])

print('\nData types after correction:')
print(fraud.dtypes)

## 2. Univariate Distributions

In [ ]:
num_cols = ['purchase_value', 'age']
fig, axes = plt.subplots(1, len(num_cols), figsize=(12, 4))
for ax, col in zip(axes, num_cols):
    fraud[col].hist(bins=40, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

cat_cols = ['source', 'browser', 'sex']
fig, axes = plt.subplots(1, len(cat_cols), figsize=(14, 4))
for ax, col in zip(axes, cat_cols):
    fraud[col].value_counts().plot(kind='bar', ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 3. Class Imbalance

In [ ]:
class_dist = fraud['class'].value_counts()
print('Class distribution:')
print(class_dist)
print(f'\nFraud rate: {class_dist[1] / len(fraud):.2%}')

class_dist.plot(kind='bar', color=['steelblue', 'salmon'])
plt.xticks([0, 1], ['Legit', 'Fraud'], rotation=0)
plt.title('Class Distribution')
plt.ylabel('Count')
plt.show()

## 4. Bivariate Analysis

In [ ]:
# Purchase value by class
fraud.groupby('class')['purchase_value'].plot(kind='kde', legend=True)
plt.title('Purchase Value by Class')
plt.xlabel('Purchase Value')
plt.show()

# Fraud rate by source and browser
for col in ['source', 'browser', 'sex']:
    rate = fraud.groupby(col)['class'].mean().sort_values(ascending=False)
    rate.plot(kind='bar', title=f'Fraud Rate by {col}')
    plt.ylabel('Fraud Rate')
    plt.tight_layout()
    plt.show()

## 5. Geolocation Integration

In [ ]:
# Convert IP to integer
def ip_to_int(ip):
    try:
        parts = str(ip).split('.')
        return int(parts[0]) * 16777216 + int(parts[1]) * 65536 + int(parts[2]) * 256 + int(parts[3])
    except:
        return np.nan

fraud['ip_int'] = fraud['ip_address'].apply(ip_to_int)
ip_country = ip_country.rename(columns={'lower_bound_ip_address': 'lower', 'upper_bound_ip_address': 'upper'})

# Range-based merge using merge_asof
fraud_sorted = fraud.sort_values('ip_int').reset_index(drop=True)
ip_sorted = ip_country.sort_values('lower').reset_index(drop=True)

merged = pd.merge_asof(
    fraud_sorted,
    ip_sorted[['lower', 'upper', 'country']],
    left_on='ip_int',
    right_on='lower',
    direction='backward'
)

# Keep only valid range matches
merged['country'] = merged.apply(
    lambda r: r['country'] if pd.notna(r['upper']) and r['ip_int'] <= r['upper'] else 'Unknown', axis=1
)

print(f'Country assigned: {(merged["country"] != "Unknown").mean():.2%}')
merged['country'].value_counts().head(10)

In [ ]:
# Fraud rate by country (top 15)
country_fraud = merged.groupby('country').agg(
    fraud_rate=('class', 'mean'),
    count=('class', 'count')
).query('count >= 100').sort_values('fraud_rate', ascending=False).head(15)

country_fraud['fraud_rate'].plot(kind='bar', figsize=(12, 4), title='Fraud Rate by Country (min 100 txns)')
plt.ylabel('Fraud Rate')
plt.tight_layout()
plt.show()

In [ ]:
# Save enriched dataset
merged.drop(columns=['lower', 'upper'], inplace=True)
merged.to_csv('../data/processed/fraud_with_country.csv', index=False)
print('Saved: data/processed/fraud_with_country.csv')